In [ ]:
import logging

from pyspark.sql import SparkSession
from pyspark.sql.types import (StructType, StructField, StringType, LongType, 
                                IntegerType, DoubleType, DateType, TimestampType, DecimalType, BooleanType)

In [ ]:
# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
try:
    spark = SparkSession.builder \
                        .appName("ECommerce ETL") \
                        .config("spark.jars", "./../../assets/libs/azure-storage-8.6.6.jar,./../../assets/libs/hadoop-azure-3.4.1.jar") \
                        .getOrCreate()
except Exception as error: 
    logger.error(error)
    spark = SparkSession.builder \
                        .appName("ECommerce ETL") \
                        .config("spark.jars.packages", "com.globalmentor:hadoop-bare-naked-local-fs") \
                        .config("spark.jars", "./../../assets/libs/azure-storage-8.6.6.jar") \
                        .config("spark.hadoop.fs.file.impl", "BareLocalFileSystem") \
                        .master("local") \
                        .getOrCreate()

In [ ]:
spark

In [ ]:
storage_name = "datastore00"
storage_secret = "KI4UuOBI2SXn4SiGTYVlNGDYI8o/DA/5309yJg77cNG2BGkhb8KZbcePajp78iYjvUXsh5fsxDSG+AStohtiQw=="

spark.conf.set("fs.azure.account.key.{0}.blob.core.windows.net".format(storage_name), storage_secret)

In [0]:
# dbutils.fs.mount(source=f"wasbs://raw@{storage_name}.blob.core.windows.net", mount_point="/mnt/raw", extra_configs={f"fs.azure.account.key.{storage_name}.blob.core.windows.net": storage_secret})

In [ ]:
# Create Schema Variables
products_schema = """product_id long, product_name string, category_id int, brand string, price decimal(10, 2), 
                     cost decimal(10, 2), stock_quantity long, weight_kg decimal(8, 2), dimensions string, 
                     description string, is_active boolean, created_at timestamp
                  """
customers_schema = StructType([
                                StructField("customer_id", LongType(), False),
                                StructField("email", StringType(), False),
                                StructField("first_name", StringType(), False),
                                StructField("last_name", StringType(), False),
                                StructField("phone", StringType(), True),
                                StructField("date_of_birth", DateType(), False),
                                StructField("gender", StringType(), False),
                                StructField("country", StringType(), False),
                                StructField("city", StringType(), False),
                                StructField("postal_code", StringType(), False),
                                StructField("address", StringType(), False),
                                StructField("registration_date", TimestampType(), False),
                                StructField("last_login", TimestampType(), False),
                                StructField("is_active", BooleanType(), False),
                                StructField("customer_segment", StringType(), False),
                                StructField("marketing_consent", BooleanType(), False)
                            ])
orders_schema = StructType([
                            StructField("order_id", LongType(), False),
                            StructField("customer_id", IntegerType(), False),
                            StructField("order_date", TimestampType(), False),
                            StructField("status", StringType(), False),
                            StructField("payment_method", StringType(), False),
                            StructField("shipping_address", StringType(), False),
                            StructField("billing_address", StringType(), False),
                            StructField("discount_amount", DecimalType(10, 2), False),
                            StructField("tax_amount", DecimalType(10, 2), False),
                            StructField("shipping_cost", DecimalType(10, 2), False),
                            StructField("total_amount", DecimalType(10, 2), False),
                            StructField("subtotal", DecimalType(10, 2), False),
                            StructField("currency", StringType(), False),
                            StructField("created_at", TimestampType(), False),
                            StructField("updated_at", TimestampType(), False)
                        ])
order_items_schema = StructType([
                                StructField("order_item_id", LongType(), False),
                                StructField("order_id", IntegerType(), False),
                                StructField("product_id", IntegerType(), False),
                                StructField("quantity", IntegerType(), False),
                                StructField("unit_price", DecimalType(10, 2), False),
                                StructField("line_total", DecimalType(10, 2), False),
                                StructField("discount_amount", DecimalType(10, 2), False)
                            ])
reviews_schema = StructType([
                            StructField("review_id", LongType(), False),
                            StructField("customer_id", IntegerType(), False),
                            StructField("product_id", IntegerType(), False),
                            StructField("rating", IntegerType(), False),
                            StructField("title", StringType(), False),
                            StructField("comment", StringType(), True),
                            StructField("is_verified_purchase", BooleanType(), False),
                            StructField("helpful_votes", IntegerType(), False),
                            StructField("created_at", TimestampType(), False)
                        ])
inventory_logs_schema = StructType([
                                    StructField("log_id", LongType(), False),
                                    StructField("product_id", IntegerType(), False),
                                    StructField("movement_type", StringType(), False),
                                    StructField("quantity_change", IntegerType(), False),
                                    StructField("reason", StringType(), False),
                                    StructField("timestamp", TimestampType(), False),
                                    StructField("reference_id", IntegerType(), False),
                                    StructField("notes", StringType(), True)
                                ])

In [ ]:
raw_input_path = f"wasbs://raw@{storage_name}.blob.core.windows.net"

In [ ]:
# Load a CSV file into a DataFrame
categories_df = spark.read.csv(raw_input_path + "/ecommerce/categories.csv", header=True, inferSchema=True, timestampFormat="yyyy-MM-dd'T'HH:mm:ss", dateFormat="yyyy-MM-dd")
products_df = spark.read.csv(raw_input_path + "/ecommerce/products.csv", header=True, schema=products_schema, timestampFormat="yyyy-MM-dd'T'HH:mm:ss", dateFormat="yyyy-MM-dd")
customers_df = spark.read.csv(raw_input_path + "/ecommerce/customers.csv", header=True, schema=customers_schema, timestampFormat="yyyy-MM-dd'T'HH:mm:ss", dateFormat="yyyy-MM-dd")
orders_df = spark.read.csv(raw_input_path + "/ecommerce/orders*.csv", header=True, schema=orders_schema, timestampFormat="yyyy-MM-dd'T'HH:mm:ss", dateFormat="yyyy-MM-dd")
order_items_df = spark.read.csv(raw_input_path + "/ecommerce/order_items.csv", header=True, schema=order_items_schema, timestampFormat="yyyy-MM-dd'T'HH:mm:ss", dateFormat="yyyy-MM-dd")
reviews_df = spark.read.csv(raw_input_path + "/ecommerce/reviews*.csv", header=True, schema=reviews_schema, timestampFormat="yyyy-MM-dd'T'HH:mm:ss", dateFormat="yyyy-MM-dd")
inventory_logs_df = spark.read.csv(raw_input_path + "/ecommerce/inventory_logs.csv", header=True, schema=inventory_logs_schema, timestampFormat="yyyy-MM-dd'T'HH:mm:ss", dateFormat="yyyy-MM-dd")


In [0]:
# Print Schemas
categories_df.printSchema()
products_df.printSchema()
customers_df.printSchema()
orders_df.printSchema()
order_items_df.printSchema()
reviews_df.printSchema()
inventory_logs_df.printSchema()

In [0]:
# %ls /mnt/raw/ecommerce/massive